# K-means Clustering

In this exercise, we will implement the K-means algorithm and use it for image compression.

* We will start with a sample dataset that will help us gain an intuition of how the K-means algorithm works.
* After that, we will use the K-means algorithm for image compression by reducing the number of colors that occur in an image to only those that are most common in that image.




# Outline
- [ 1 - Implementing K-means](#1)
  - [ 1.1 Finding closest centroids](#1.1)
    - [ Exercise 1](#ex01)
  - [ 1.2 Computing centroid means](#1.2)
    - [ Exercise 2](#ex02)
- [ 2 - K-means on a sample dataset ](#2)
- [ 3 - Random initialization](#3)
- [ 4 - Image compression with K-means](#4)
  - [ 4.1 Dataset](#4.1)
  - [ 4.2 K-Means on image pixels](#4.2)
  - [ 4.3 Compress the image](#4.3)


First, run the cell below to import the packages needed in this assignment:

- [numpy](https://numpy.org/) is the fundamental package for scientific computing with Python.
- [matplotlib](http://matplotlib.org) is a popular library to plot graphs in Python.
- `utils.py` contains helper functions for this assignment.

In [1]:
%%writefile utils.py
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def load_data():
    """
    Loads an example dataset for K-means.
    Returns:
        X (ndarray): (m, n) Input values
    """
    # A sample 2D dataset for K-means demonstration
    data = np.array([
        [1.842079, 4.607572], [5.658514, 4.799964], [6.352579, 3.290854],
        [4.838139, 4.630040], [4.945088, 4.794406], [6.002131, 3.293093],
        [3.036720, 5.097072], [6.839843, 3.328325], [4.747970, 4.542738],
        [5.309062, 4.721491], [7.712610, 3.473523], [4.686566, 4.521838],
        [3.921160, 4.962059], [4.200585, 4.547746], [4.331206, 4.869032],
        [3.945920, 5.242859], [4.697412, 4.537676], [5.361955, 4.908298],
        [5.723145, 3.518606], [4.745423, 4.453303], [4.492576, 4.873244],
        [5.395724, 4.796792], [5.372579, 3.298197], [5.321677, 4.551608],
        [4.898863, 4.676646], [4.269018, 5.086422], [5.973412, 4.814324],
        [5.176465, 3.342981], [5.544158, 4.847154], [5.405786, 3.333036]
    ])
    return data

def plot_progress_kMeans(X, centroids, previous_centroids, idx, K, i):
    """
    Plots the data points and centroids, showing the progress of K-Means.
    """
    colors = plt.cm.get_cmap('viridis', K)
    plt.scatter(X[:, 0], X[:, 1], c=idx, cmap=colors, s=30, alpha=0.8)

    # Plot the centroids as black X's
    plt.scatter(centroids[:, 0], centroids[:, 1], marker='x', s=100, linewidths=3, color='black', label='Centroids')

    # Plot the history of the centroids with a dotted line
    for j in range(centroids.shape[0]):
        plt.plot([previous_centroids[j, 0], centroids[j, 0]],
                 [previous_centroids[j, 1], centroids[j, 1]],
                 'k--', markersize=10, linewidth=1)

    plt.title(f'Iteration {i}')
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.legend()

def plot_kMeans_RGB(X_img, centroids, idx, K):
    """
    Plots the image pixels in RGB space and marks the centroids.
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Scatter plot of all pixels
    ax.scatter(X_img[:, 0], X_img[:, 1], X_img[:, 2], c=X_img, s=1)

    # Plot centroids
    ax.scatter(centroids[:, 0], centroids[:, 1], centroids[:, 2],
               marker='x', s=200, linewidths=3, color='red', label='Centroids')

    ax.set_xlabel('Red')
    ax.set_ylabel('Green')
    ax.set_zlabel('Blue')
    ax.set_title(f'Pixel Colors and {K} Centroids in RGB Space')
    ax.legend()
    plt.show()

def show_centroid_colors(centroids):
    """
    Visualizes the centroid colors as swatches.
    """
    K = centroids.shape[0]
    fig, ax = plt.subplots(1, K, figsize=(K * 1.5, 2))
    for i in range(K):
        ax[i].imshow(centroids[i].reshape(1, 1, 3))
        ax[i].set_title(f'Idx: {i}')
        ax[i].axis('off')
    plt.suptitle('Centroid Colors')
    plt.tight_layout(rect=[0, 0.03, 1, 0.9])
    plt.show()


Writing utils.py


In [2]:
from utils import *
%matplotlib inline

<a name="1"></a>
## 1 - Implementing K-means

The K-means algorithm is a method to automatically cluster similar
data points together.

* Concretely, you are given a training set $\{x^{(1)}, ..., x^{(m)}\}$, and you want
to group the data into a few cohesive “clusters”.


* K-means is an iterative procedure that
     * Starts by guessing the initial centroids, and then
     * Refines this guess by
         * Repeatedly assigning examples to their closest centroids, and then
         * Recomputing the centroids based on the assignments.
         

* In pseudocode, the K-means algorithm is as follows:

    ``` python
    # Initialize centroids
    # K is the number of clusters
    centroids = kMeans_init_centroids(X, K)
    
    for iter in range(iterations):
        # Cluster assignment step:
        # Assign each data point to the closest centroid.
        # idx[i] corresponds to the index of the centroid
        # assigned to example i
        idx = find_closest_centroids(X, centroids)

        # Move centroid step:
        # Compute means based on centroid assignments
        centroids = compute_centroids(X, idx, K)
    ```


* The inner-loop of the algorithm repeatedly carries out two steps:
    1. Assigning each training example $x^{(i)}$ to its closest centroid, and
    2. Recomputing the mean of each centroid using the points assigned to it.
    
    
* The $K$-means algorithm will always converge to some final set of means for the centroids.

* However, the converged solution may not always be ideal and depends on the initial setting of the centroids.
    * Therefore, in practice the K-means algorithm is usually run a few times with different random initializations.
    * One way to choose between these different solutions from different random initializations is to choose the one with the lowest cost function value (distortion).

We will implement the two phases of the K-means algorithm separately
in the next sections.
* We will start by completing `find_closest_centroid` and then proceed to complete `compute_centroids`.